In [2]:
# ==========================================================
#                  CELL 1: ENVIRONMENT SETUP & PATH DEFINITION
# ==========================================================
import os
import torch

# 🛑 ABSOLUTE PATH TO YOUR DATASET ROOT 🛑
# This path contains the 'train', 'testA', and 'testB' folders.
# Windows paths are used here:
DATASET_ROOT = "D:/grad-project/amazon-dataset" 

# Define the device (Will use GPU if available, otherwise CPU)
# Ensure you have set up PyTorch with CUDA if you want to use your GPU.
device = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")

# Define the paths for saving and loading the model/scalers
# These files will be saved in the directory where your VS Code script is running.
OUTPUT_SCALER_PATH = 'output_scaler.joblib'
BEST_MODEL_PATH_EXP3 = 'mnasnet_best_norm.pt' 
BEST_MODEL_PATH_EXP4 = 'mnasnet_best_aug.pt' 

print(f"Data root defined as: {DATASET_ROOT}")
print(f"Device set to: {device}")
print("✅ Environment setup complete.")

Data root defined as: D:/grad-project/amazon-dataset
Device set to: cpu
✅ Environment setup complete.


In [ ]:
# ==========================================================
#          CELL 2: DEFINE FOLDERS & LOAD MODEL
# ==========================================================
import torch
import torchvision.models as models
import os
import time

print("Using dataset folders directly from Kaggle input...")

# 1. Define paths inside the Kaggle dataset
DATASET_ROOT = "/kaggle/input/amazon-dataset/amazon-dataset"  # adjust if needed

train_dir = os.path.join(DATASET_ROOT, "train")
testA_dir = os.path.join(DATASET_ROOT, "testA")
testB_dir = os.path.join(DATASET_ROOT, "testB")

print("Train folder path :", train_dir)
print("TestA folder path :", testA_dir)
print("TestB folder path :", testB_dir)

print("\nContents of train folder (first few entries):")
print(os.listdir(train_dir)[:10])

# 2. Load your MnasNet model WITHOUT internet (no pretrained weights)
mnasnet = models.mnasnet1_0(weights=None)
print("\n--- MnasNet Model Architecture Loaded (no pretrained weights). ---")


: 

In [9]:
# ==========================================================
#                  MODEL SUMMARY CELL (KAGGLE, OFFLINE)
# ==========================================================
import torch
import torchvision.models as models

# Load the MnasNet model WITHOUT pretrained weights (offline-friendly)
mnasnet = models.mnasnet1_0(weights=None)  # or pretrained=False for older versions

print("\n\n--- MnasNet Model Architecture (no pretrained weights) ---")
print(mnasnet)




--- MnasNet Model Architecture (no pretrained weights) ---
MNASNet(
  (layers): Sequential(
    (0): Conv2d(3, 32, kernel_size=(3, 3), stride=(2, 2), padding=(1, 1), bias=False)
    (1): BatchNorm2d(32, eps=1e-05, momentum=0.00029999999999996696, affine=True, track_running_stats=True)
    (2): ReLU(inplace=True)
    (3): Conv2d(32, 32, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), groups=32, bias=False)
    (4): BatchNorm2d(32, eps=1e-05, momentum=0.00029999999999996696, affine=True, track_running_stats=True)
    (5): ReLU(inplace=True)
    (6): Conv2d(32, 16, kernel_size=(1, 1), stride=(1, 1), bias=False)
    (7): BatchNorm2d(16, eps=1e-05, momentum=0.00029999999999996696, affine=True, track_running_stats=True)
    (8): Sequential(
      (0): _InvertedResidual(
        (layers): Sequential(
          (0): Conv2d(16, 48, kernel_size=(1, 1), stride=(1, 1), bias=False)
          (1): BatchNorm2d(48, eps=1e-05, momentum=0.00029999999999996696, affine=True, track_running_stats=True)

In [10]:
import torch
import torch.nn as nn
import torchvision.models as models

# 1. Load the MnasNet engine WITHOUT pretrained weights (offline)
mnasnet = models.mnasnet1_0(weights=None)  # IMPORTANT: no internet download

# 2. Freeze the engine (backbone)
for param in mnasnet.parameters():
    param.requires_grad = False

# 3. Perform the "head replacement"
print("Original head (classifier):", mnasnet.classifier)

# Get number of input features to the classifier
num_input_features = mnasnet.classifier[1].in_features

# Replace with custom head for 14 outputs
mnasnet.classifier[1] = nn.Linear(num_input_features, 14)

# 4. Verify the swap
print("\n--- Model After Head Replacement ---")
print(mnasnet.classifier)


Original head (classifier): Sequential(
  (0): Dropout(p=0.2, inplace=True)
  (1): Linear(in_features=1280, out_features=1000, bias=True)
)

--- Model After Head Replacement ---
Sequential(
  (0): Dropout(p=0.2, inplace=True)
  (1): Linear(in_features=1280, out_features=14, bias=True)
)


In [11]:
# ==========================================================
# 🛑 CORRECTED DATASET AND DATALOADER CELL (KAGGLE) 🛑
# ==========================================================
import torch
from torch.utils.data import Dataset, DataLoader
import pandas as pd
from PIL import Image
from torchvision import transforms
import os
import numpy as np
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split

# 1. Define data paths (inside Kaggle dataset)
DATASET_ROOT = "/kaggle/input/amazon-dataset/amazon-dataset"  # adjust if needed
data_root = os.path.join(DATASET_ROOT, "train")
images_dir_front = os.path.join(data_root, "mask")
images_dir_side = os.path.join(data_root, "mask_left")

class BodyMDataset(Dataset):
    def __init__(self, root_dir, transform=None):
        self.root_dir = root_dir
        self.transform = transform

        # Image dirs
        self.images_dir_front = os.path.join(root_dir, "mask")
        self.images_dir_side = os.path.join(root_dir, "mask_left")

        # CSV paths
        measurements_path = os.path.join(root_dir, "measurements.csv")
        hwg_path = os.path.join(root_dir, "hwg_metadata.csv")
        photo_map_path = os.path.join(root_dir, "subject_to_photo_map.csv")

        measurements_df = pd.read_csv(measurements_path)
        hwg_df = pd.read_csv(hwg_path)
        photo_map_df = pd.read_csv(photo_map_path)

        # Filter for subjects with at least two photos
        photo_counts = photo_map_df["subject_id"].value_counts()
        subjects_with_two_photos = photo_counts[photo_counts >= 2].index.tolist()

        master_df_unfiltered = pd.merge(measurements_df, hwg_df, on="subject_id")

        self.master_df = master_df_unfiltered[
            master_df_unfiltered["subject_id"].isin(subjects_with_two_photos)
        ].reset_index(drop=True)
        self.photo_map_df = photo_map_df[
            photo_map_df["subject_id"].isin(subjects_with_two_photos)
        ].reset_index(drop=True)

    def __len__(self):
        return len(self.master_df)

    def __getitem__(self, idx):
        data_row = self.master_df.iloc[idx]
        subject_id = data_row["subject_id"]

        photo_ids = self.photo_map_df[
            self.photo_map_df["subject_id"] == subject_id
        ]["photo_id"].tolist()

        if len(photo_ids) < 2:
            return None

        photo_id_front = photo_ids[0]
        photo_id_side = photo_ids[1]

        img_path_front = os.path.join(self.images_dir_front, f"{photo_id_front}.png")
        img_path_side = os.path.join(self.images_dir_side, f"{photo_id_side}.png")

        try:
            image_front = Image.open(img_path_front).convert("RGB")
            image_side = Image.open(img_path_side).convert("RGB")
        except FileNotFoundError as e:
            print(f"Error loading image for subject {subject_id}. Check file path: {e}")
            raise e

        height = data_row["height_cm"]
        weight = data_row["weight_kg"]

        measurement_cols = [
            "ankle", "arm-length", "bicep", "calf", "chest", "forearm", "height",
            "hip", "leg-length", "shoulder-breadth", "shoulder-to-crotch",
            "thigh", "waist", "wrist"
        ]
        measurements = data_row[measurement_cols].values.astype("float32")
        measurements = torch.from_numpy(measurements)

        if self.transform:
            image_front = self.transform(image_front)
            image_side = self.transform(image_side)

        sample = {
            "subject_id": subject_id,
            "image_front": image_front,
            "image_side": image_side,
            "height": torch.tensor(height, dtype=torch.float32),
            "weight": torch.tensor(weight, dtype=torch.float32),
            "measurements": measurements,
        }
        return sample

# 3. Transforms
data_transforms = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406],
                         std=[0.229, 0.224, 0.225]),
])

# 4. Create dataset and loaders
print(f"Attempting to load data from: {data_root}")

try:
    full_dataset = BodyMDataset(root_dir=data_root, transform=data_transforms)

    train_size = int(0.8 * len(full_dataset))
    val_size = len(full_dataset) - train_size

    train_dataset, val_dataset = torch.utils.data.random_split(
        full_dataset, [train_size, val_size]
    )

    train_dataloader = DataLoader(train_dataset, batch_size=32, shuffle=True, num_workers=2)
    val_dataloader = DataLoader(val_dataset, batch_size=32, shuffle=False, num_workers=2)

    print(f"✅ Successfully created dataset with {len(full_dataset)} samples (after filtering).")
    print(f"   -> Train samples: {len(train_dataset)}, Validation samples: {len(val_dataset)}")

    first_sample = full_dataset[0]
    print("\n--- Testing with one sample ---")
    print("Sample Subject ID:", first_sample["subject_id"])
    print("Front Image Tensor Shape:", first_sample["image_front"].shape)
    print("-----------------------------")

    print("✅ Successfully created train_dataloader and val_dataloader.")
except Exception as e:
    print(f"❌ A CRITICAL error occurred: {e}")


Attempting to load data from: /kaggle/input/amazon-dataset/amazon-dataset/train
✅ Successfully created dataset with 1327 samples (after filtering).
   -> Train samples: 1061, Validation samples: 266

--- Testing with one sample ---
Sample Subject ID: -494U-YoXOD8e8gkCuyaRLn4MLo5P8Dm2B1s59WBGdg
Front Image Tensor Shape: torch.Size([3, 224, 224])
-----------------------------
✅ Successfully created train_dataloader and val_dataloader.


In [12]:
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
import pandas as pd
from PIL import Image
from torchvision import transforms
import os

DATASET_ROOT = "/kaggle/input/amazon-dataset/amazon-dataset"  # adjust if needed

data_transforms = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406],
                         std=[0.229, 0.224, 0.225]),
])

class BodyMDataset(Dataset):
    def __init__(self, root_dir, transform=None):
        self.root_dir = root_dir
        self.transform = transform

        measurements_path = os.path.join(root_dir, "measurements.csv")
        hwg_path = os.path.join(root_dir, "hwg_metadata.csv")
        photo_map_path = os.path.join(root_dir, "subject_to_photo_map.csv")

        measurements_df = pd.read_csv(measurements_path)
        hwg_df = pd.read_csv(hwg_path)
        photo_map_df = pd.read_csv(photo_map_path)

        photo_counts = photo_map_df["subject_id"].value_counts()
        subjects_with_two_photos = photo_counts[photo_counts >= 2].index.tolist()

        master_df_unfiltered = pd.merge(measurements_df, hwg_df, on="subject_id")

        self.master_df = master_df_unfiltered[
            master_df_unfiltered["subject_id"].isin(subjects_with_two_photos)
        ]
        self.photo_map_df = photo_map_df[
            photo_map_df["subject_id"].isin(subjects_with_two_photos)
        ]

    def __len__(self):
        return len(self.master_df)

    def __getitem__(self, idx):
        data_row = self.master_df.iloc[idx]
        subject_id = data_row["subject_id"]
        photo_ids = self.photo_map_df[
            self.photo_map_df["subject_id"] == subject_id
        ]["photo_id"].tolist()

        photo_id_front, photo_id_side = photo_ids[0], photo_ids[1]

        img_path_front = os.path.join(self.root_dir, "mask", f"{photo_id_front}.png")
        img_path_side = os.path.join(self.root_dir, "mask_left", f"{photo_id_side}.png")

        image_front = Image.open(img_path_front).convert("RGB")
        image_side = Image.open(img_path_side).convert("RGB")

        height = data_row["height_cm"]
        weight = data_row["weight_kg"]
        measurement_cols = [
            "ankle", "arm-length", "bicep", "calf", "chest", "forearm", "height",
            "hip", "leg-length", "shoulder-breadth", "shoulder-to-crotch",
            "thigh", "waist", "wrist"
        ]
        measurements = data_row[measurement_cols].values.astype("float32")
        measurements = torch.from_numpy(measurements)

        if self.transform:
            image_front = self.transform(image_front)
            image_side = self.transform(image_side)

        sample = {
            "subject_id": subject_id,
            "image_front": image_front,
            "image_side": image_side,
            "height": torch.tensor(height, dtype=torch.float32),
            "weight": torch.tensor(weight, dtype=torch.float32),
            "measurements": measurements,
        }
        return sample

# Paths for testA and testB inside Kaggle dataset
testA_root = os.path.join(DATASET_ROOT, "testA")
testB_root = os.path.join(DATASET_ROOT, "testB")

testA_dataloader, testB_dataloader = None, None

if os.path.exists(testA_root):
    try:
        testA_dataset = BodyMDataset(root_dir=testA_root, transform=data_transforms)
        testA_dataloader = DataLoader(testA_dataset, batch_size=32, shuffle=False)
        print(f"\n✅ Successfully created TestA DataLoader with {len(testA_dataset)} samples.")
    except Exception as e:
        print(f"❌ Error loading TestA dataset from {testA_root}: {e}")
else:
    print(f"❌ Could not find TestA directory at '{testA_root}'.")

if os.path.exists(testB_root):
    try:
        testB_dataset = BodyMDataset(root_dir=testB_root, transform=data_transforms)
        testB_dataloader = DataLoader(testB_dataset, batch_size=32, shuffle=False)
        print(f"✅ Successfully created TestB DataLoader with {len(testB_dataset)} samples.")
    except Exception as e:
        print(f"❌ Error loading TestB dataset from {testB_root}: {e}")
else:
    print(f"❌ Could not find TestB directory at '{testB_root}'.")

def evaluate_model(model, dataloader, criterion, device):
    model.eval()
    total_loss = 0.0
    num_samples = 0
    with torch.no_grad():
        for batch in dataloader:
            front_images = batch["image_front"].to(device)
            measurements = batch["measurements"].to(device)
            outputs = model(front_images)
            loss = criterion(outputs, measurements)
            total_loss += loss.item() * front_images.size(0)
            num_samples += front_images.size(0)
    return total_loss / num_samples if num_samples > 0 else 0

print("\n--- Ready for Evaluation ---")



✅ Successfully created TestA DataLoader with 87 samples.
✅ Successfully created TestB DataLoader with 389 samples.

--- Ready for Evaluation ---


In [13]:
import torch
import torch.nn as nn
import torchvision.models as models

class MultiInputMnasNet(nn.Module):
    def __init__(self):
        super(MultiInputMnasNet, self).__init__()

        # Image processor: MnasNet WITHOUT pretrained weights (offline)
        mnasnet_base = models.mnasnet1_0(weights=None)
        self.features = mnasnet_base.layers

        for param in self.features.parameters():
            param.requires_grad = False

        self.numerical_processor = nn.Sequential(
            nn.Linear(2, 16),
            nn.ReLU(),
            nn.Linear(16, 32),
            nn.ReLU(),
        )

        self.classifier = nn.Linear(2592, 14)

    def forward(self, image_front, image_side, height, weight):
        x1 = self.features(image_front)
        x1 = x1.mean([2, 3])

        x2 = self.features(image_side)
        x2 = x2.mean([2, 3])

        numerical_input = torch.cat([height.unsqueeze(1), weight.unsqueeze(1)], dim=1)
        x3 = self.numerical_processor(numerical_input)

        combined = torch.cat([x1, x2, x3], dim=1)
        output = self.classifier(combined)
        return output

multi_input_model = MultiInputMnasNet()
print("--- New Multi-Input Model Architecture ---")
print(multi_input_model)


--- New Multi-Input Model Architecture ---
MultiInputMnasNet(
  (features): Sequential(
    (0): Conv2d(3, 32, kernel_size=(3, 3), stride=(2, 2), padding=(1, 1), bias=False)
    (1): BatchNorm2d(32, eps=1e-05, momentum=0.00029999999999996696, affine=True, track_running_stats=True)
    (2): ReLU(inplace=True)
    (3): Conv2d(32, 32, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), groups=32, bias=False)
    (4): BatchNorm2d(32, eps=1e-05, momentum=0.00029999999999996696, affine=True, track_running_stats=True)
    (5): ReLU(inplace=True)
    (6): Conv2d(32, 16, kernel_size=(1, 1), stride=(1, 1), bias=False)
    (7): BatchNorm2d(16, eps=1e-05, momentum=0.00029999999999996696, affine=True, track_running_stats=True)
    (8): Sequential(
      (0): _InvertedResidual(
        (layers): Sequential(
          (0): Conv2d(16, 48, kernel_size=(1, 1), stride=(1, 1), bias=False)
          (1): BatchNorm2d(48, eps=1e-05, momentum=0.00029999999999996696, affine=True, track_running_stats=True)
     

In [14]:
# ==========================================================
#                  CELL A: CREATE & SPLIT DATA (KAGGLE)
# ==========================================================
from torch.utils.data import DataLoader, random_split
import torch

DATASET_ROOT = "/kaggle/input/amazon-dataset/amazon-dataset"  # adjust if needed
data_root = os.path.join(DATASET_ROOT, "train")

try:
    full_dataset = BodyMDataset(root_dir=data_root, transform=data_transforms)
    print(f"✅ Successfully created full_dataset with {len(full_dataset)} samples.")
except Exception as e:
    print(f"❌ An error occurred creating full_dataset: {e}")
    print("Please make sure the 'BodyMDataset' class is defined above.")
    raise

train_size = int(0.8 * len(full_dataset))
val_size = len(full_dataset) - train_size

train_dataset, val_dataset = random_split(full_dataset, [train_size, val_size])

print(f"\nDataset split successfully:")
print(f"Total samples:      {len(full_dataset)}")
print(f"Training samples:   {len(train_dataset)}")
print(f"Validation samples: {len(val_dataset)}")

batch_size = 32
train_dataloader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True, num_workers=2)
val_dataloader = DataLoader(val_dataset, batch_size=batch_size, shuffle=False, num_workers=2)

print("\n✅ train_dataloader and val_dataloader are ready!")


✅ Successfully created full_dataset with 1327 samples.

Dataset split successfully:
Total samples:      1327
Training samples:   1061
Validation samples: 266

✅ train_dataloader and val_dataloader are ready!


In [ ]:
# ==========================================================
#                  CELL B: TRAIN THE MODEL (with Early Stopping)
# ==========================================================
import torch  # <--- THIS WAS MISSING
import torch.optim as optim
import torch.nn as nn
import numpy as np
import time

# --- 1. Set up the Training Tools ---

# Move the new model to the correct device
# On Kaggle, ensure you have enabled GPU in the Notebook settings (Sidebar -> Accelerator -> GPU)
device = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")
multi_input_model.to(device)

# The "Taste Tester" - Mean Squared Error Loss
criterion = nn.MSELoss()

# The "Optimizer"
optimizer = optim.Adam(list(multi_input_model.numerical_processor.parameters()) + list(multi_input_model.classifier.parameters()), lr=0.001)

# --- Early Stopping Setup ---
num_epochs = 20 # We can set this higher, early stopping will find the best one
patience = 3    # How many epochs to wait for improvement before stopping
patience_counter = 0
best_val_loss = np.inf
best_model_path = 'best_multi_input_model.pth' # Path to save the best model
# ---------------------------------

print(f"Starting training for the multi-input model on {device}...")
print(f"Will stop if validation loss does not improve for {patience} epochs.")

# --- 2. The New Training Loop ---
for epoch in range(num_epochs):

    start_time = time.time()
    multi_input_model.train() # Set the model to training mode
    running_loss = 0.0

    # === TRAINING PHASE ===
    for i, batch in enumerate(train_dataloader):
        # Get ALL the inputs and move them to the device
        front_images = batch['image_front'].to(device)
        side_images = batch['image_side'].to(device)
        heights = batch['height'].to(device)
        weights = batch['weight'].to(device)
        measurements = batch['measurements'].to(device)

        optimizer.zero_grad()
        outputs = multi_input_model(front_images, side_images, heights, weights)
        loss = criterion(outputs, measurements)
        loss.backward()
        optimizer.step()
        running_loss += loss.item()

    epoch_train_loss = running_loss / len(train_dataloader)

    # === VALIDATION PHASE ===
    multi_input_model.eval() # Set the model to evaluation mode
    val_loss = 0.0
    with torch.no_grad(): # Tell PyTorch not to calculate gradients
        # This line will now work
        for batch in val_dataloader:
            # Get inputs
            front_images = batch['image_front'].to(device)
            side_images = batch['image_side'].to(device)
            heights = batch['height'].to(device)
            weights = batch['weight'].to(device)
            measurements = batch['measurements'].to(device)

            outputs = multi_input_model(front_images, side_images, heights, weights)
            loss = criterion(outputs, measurements)
            val_loss += loss.item()

    epoch_val_loss = val_loss / len(val_dataloader)
    epoch_time = time.time() - start_time

    print(f'Epoch {epoch + 1}/{num_epochs} | Time: {epoch_time:.2f}s | Train Loss: {epoch_train_loss:.3f} | Validation Loss: {epoch_val_loss:.3f}')

    # === Early Stopping Logic ===
    if epoch_val_loss < best_val_loss:
        print(f'Validation loss improved ({best_val_loss:.3f} --> {epoch_val_loss:.3f}). Saving model... 💾')
        best_val_loss = epoch_val_loss
        patience_counter = 0 # Reset patience
        torch.save(multi_input_model.state_dict(), best_model_path)
    else:
        patience_counter += 1
        print(f'Validation loss did not improve. Patience {patience_counter}/{patience}')

    if patience_counter >= patience:
        print(f'\n--- 🛑 Early stopping triggered after {epoch + 1} epochs! ---')
        break # Exit the training loop

# --- 3. Load the Best Model ---
print(f"\nTraining complete. Loading best model from {best_model_path}...")
multi_input_model.load_state_dict(torch.load(best_model_path))
print("✅ Best model loaded! Ready for final evaluation.")

Starting training for the multi-input model on cpu...
Will stop if validation loss does not improve for 3 epochs.
Epoch 1/20 | Time: 110.10s | Train Loss: 3095.351 | Validation Loss: 2217.043
Validation loss improved (inf --> 2217.043). Saving model... 💾
Epoch 2/20 | Time: 106.69s | Train Loss: 310.967 | Validation Loss: 787.189
Validation loss improved (2217.043 --> 787.189). Saving model... 💾
Epoch 3/20 | Time: 107.53s | Train Loss: 26.915 | Validation Loss: 732.038
Validation loss improved (787.189 --> 732.038). Saving model... 💾
Epoch 4/20 | Time: 107.21s | Train Loss: 21.729 | Validation Loss: 726.973
Validation loss improved (732.038 --> 726.973). Saving model... 💾
Epoch 5/20 | Time: 107.55s | Train Loss: 20.817 | Validation Loss: 687.941
Validation loss improved (726.973 --> 687.941). Saving model... 💾
Epoch 6/20 | Time: 107.36s | Train Loss: 20.845 | Validation Loss: 658.460
Validation loss improved (687.941 --> 658.460). Saving model... 💾
Epoch 7/20 | Time: 107.67s | Train Los

In [ ]:
# --- 1. Define the Evaluation Function for the Multi-Input Model ---
def evaluate_multi_input_model(model, dataloader, criterion, device):
    model.eval()  # Set the model to evaluation mode
    total_loss = 0.0
    num_samples = 0

    with torch.no_grad():  # We don't need to calculate gradients
        for batch in dataloader:
            # Get all four inputs from the batch
            front_images = batch['image_front'].to(device)
            side_images = batch['image_side'].to(device)
            heights = batch['height'].to(device)
            weights = batch['weight'].to(device)
            measurements = batch['measurements'].to(device)

            # Pass all inputs to the model
            outputs = model(front_images, side_images, heights, weights)
            loss = criterion(outputs, measurements)

            total_loss += loss.item() * front_images.size(0)
            num_samples += front_images.size(0)

    avg_loss = total_loss / num_samples
    return avg_loss

# --- 2. Run the Final Evaluation ---
# We use the testA_dataloader and testB_dataloader created previously
print("--- Starting Final Evaluation of the Multi-Input Model ---")

if testA_dataloader and len(testA_dataloader.dataset) > 0:
    loss_testA_new = evaluate_multi_input_model(multi_input_model, testA_dataloader, criterion, device)
    print(f"Average Loss on TestA (in-distribution): {loss_testA_new:.3f}")

if testB_dataloader and len(testB_dataloader.dataset) > 0:
    loss_testB_new = evaluate_multi_input_model(multi_input_model, testB_dataloader, criterion, device)
    print(f"Average Loss on TestB (in-the-wild):    {loss_testB_new:.3f}")

# --- 3. (Optional but Recommended) Save Your Trained Model ---
# After seeing the results, you can save the model's state for future use.
# This saves the "brain" of your trained model.
model_save_path = '/content/multi_input_model_final.pth'
torch.save(multi_input_model.state_dict(), model_save_path)
print(f"\n✅ Model saved to '{model_save_path}'")

In [ ]:
# ==========================================================
#    NEW EXPERIMENT: TRAINING WITH MAE (L1 LOSS) + EARLY STOPPING
# ==========================================================
import torch
import torch.optim as optim
import torch.nn as nn
import numpy as np
import time

print("--- STARTING NEW EXPERIMENT (TRAINING WITH L1/MAE LOSS) ---")

# --- 1. Set up the Training Tools ---
# (We assume 'multi_input_model' is already created)

# Move the model to the correct device
device = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")
multi_input_model.to(device) # Move the *same model* to the device

# ▼▼▼ THE CHANGE FOR THIS EXPERIMENT ▼▼▼
# We are now using MAE Loss, which is L1Loss.
criterion = nn.L1Loss()
# ▲▲▲▲▲▲▲▲▲▲▲▲▲▲▲▲▲▲▲▲▲▲▲▲▲▲▲▲▲▲

# The "Optimizer"
# We reset the optimizer for the new training run
optimizer = optim.Adam(list(multi_input_model.numerical_processor.parameters()) + list(multi_input_model.classifier.parameters()), lr=0.001)

# --- Early Stopping Setup (INCLUDED) ---
num_epochs = 20 # Max epochs
patience = 3    # How many epochs to wait for improvement
patience_counter = 0
best_val_loss = np.inf

# ▼▼▼ We save to a NEW file to compare results ▼▼▼
best_model_path = 'best_multi_input_model_MAE.pth'
# ▲▲▲▲▲▲▲▲▲▲▲▲▲▲▲▲▲▲▲▲▲▲▲▲▲▲▲▲▲▲▲▲▲▲▲▲▲▲▲

print(f"Starting training on {device}...")
print(f"Using MAE (L1Loss) and saving best model to {best_model_path}")
print(f"Will stop if validation loss does not improve for {patience} epochs.")

# --- 2. The Training Loop ---
for epoch in range(num_epochs):

    start_time = time.time()
    multi_input_model.train()
    running_loss = 0.0

    # === TRAINING PHASE ===
    for i, batch in enumerate(train_dataloader):
        front_images = batch['image_front'].to(device)
        side_images = batch['image_side'].to(device)
        heights = batch['height'].to(device)
        weights = batch['weight'].to(device)
        measurements = batch['measurements'].to(device)

        optimizer.zero_grad()
        outputs = multi_input_model(front_images, side_images, heights, weights)
        loss = criterion(outputs, measurements)
        loss.backward()
        optimizer.step()
        running_loss += loss.item()

    epoch_train_loss = running_loss / len(train_dataloader)

    # === VALIDATION PHASE ===
    multi_input_model.eval()
    val_loss = 0.0
    with torch.no_grad():
        for batch in val_dataloader:
            front_images = batch['image_front'].to(device)
            side_images = batch['image_side'].to(device)
            heights = batch['height'].to(device)
            weights = batch['weight'].to(device)
            measurements = batch['measurements'].to(device)

            outputs = multi_input_model(front_images, side_images, heights, weights)
            loss = criterion(outputs, measurements) # This is now calculating MAE
            val_loss += loss.item()

    epoch_val_loss = val_loss / len(val_dataloader)
    epoch_time = time.time() - start_time

    print(f'Epoch {epoch + 1}/{num_epochs} | Time: {epoch_time:.2f}s | Train MAE: {epoch_train_loss:.3f} | Validation MAE: {epoch_val_loss:.3f}')

    # === Early Stopping Logic (INCLUDED) ===
    if epoch_val_loss < best_val_loss:
        print(f'Validation MAE improved ({best_val_loss:.3f} --> {epoch_val_loss:.3f}). Saving model... 💾')
        best_val_loss = epoch_val_loss
        patience_counter = 0
        torch.save(multi_input_model.state_dict(), best_model_path)
    else:
        patience_counter += 1
        print(f'Validation MAE did not improve. Patience {patience_counter}/{patience}')

    if patience_counter >= patience:
        print(f'\n--- 🛑 Early stopping triggered after {epoch + 1} epochs! ---')
        break

# --- 3. Load the Best MAE-Trained Model ---
print(f"\nTraining complete. Loading best MAE model from {best_model_path}...")
multi_input_model.load_state_dict(torch.load(best_model_path))
print("✅ Best MAE model loaded! Ready for final evaluation.")

In [ ]:
# ==========================================================
#          CELL 2: DEFINE FOLDERS & LOAD MODEL (KAGGLE FIX)
# ==========================================================
import torch
import torchvision.models as models
import os
import time

# --- 1. Define Correct Paths (Using DATASET_ROOT from CELL 1) ---
# We use the path defined in the previous cell (Kaggle dataset input)
# The local path is now the root path of the dataset itself.
DATA_ROOT = DATASET_ROOT # Use the variable defined in Cell 1

# --- 2. Verify Data Folders ---
print(f"Checking data contents at: {DATA_ROOT}")
# Expected: 'train', 'testA', 'testB'
# We don't need to copy, we use the input folder directly.
if os.path.exists(DATA_ROOT):
    print(f"✅ Data root exists.")
    print(f"Folders found: {os.listdir(DATA_ROOT)}")
else:
    print(f"🛑 ERROR: Data root not found at {DATA_ROOT}. Check Cell 1 definition.")

# --- 3. Load Model (Without External Weights) ---
print("\nLoading MnasNet model architecture...")

# We must set weights=None to prevent the model from attempting to download
# pre-trained weights from the internet, which is typically blocked on Kaggle.
mnasnet = models.mnasnet1_0(weights=None)

print("✅ MnasNet Model Architecture Loaded (randomly initialized).")

In [ ]:
# ==========================================================
#          EXPERIMENT 3 - CELL 1: CREATE SCALERS (KAGGLE FIX)
# ==========================================================
import pandas as pd
from sklearn.preprocessing import StandardScaler
import joblib
import os

print("--- Experiment 3: Normalization ---")
print("Loading training CSVs to create scalers...")

# --- 1. Load the TRAINING metadata ---
# IMPORTANT FIX: Use the DATASET_ROOT variable defined in the very first cell.
# Assuming DATASET_ROOT is: /kaggle/input/amazon-dataset/amazon-dataset
# We look for the 'train' folder inside that root.
train_data_root = os.path.join(DATASET_ROOT, 'train')

measurements_path = os.path.join(train_data_root, 'measurements.csv')
hwg_path = os.path.join(train_data_root, 'hwg_metadata.csv')

# Check if files exist before reading (for debugging)
if not os.path.exists(measurements_path):
    print(f"🛑 Error: measurements.csv not found at {measurements_path}")
    # You might need to adjust DATASET_ROOT if the structure is different
    # e.g., if files are directly in DATASET_ROOT instead of a 'train' subfolder.

measurements_df = pd.read_csv(measurements_path)
hwg_df = pd.read_csv(hwg_path)

# Merge to get all training data in one place
master_train_df = pd.merge(measurements_df, hwg_df, on='subject_id')
print(f"✅ Loaded {len(master_train_df)} training samples.")

# --- 2. Define our Input and Output columns ---
# The 2 numerical inputs
input_cols = ['height_cm', 'weight_kg']

# The 14 numerical outputs (measurements)
output_cols = [
    'ankle', 'arm-length', 'bicep', 'calf', 'chest', 'forearm', 'height',
    'hip', 'leg-length', 'shoulder-breadth', 'shoulder-to-crotch',
    'thigh', 'waist', 'wrist'
]

# --- 3. Create and "Fit" the Scalers ---
# Create one scaler for inputs
input_scaler = StandardScaler()
input_scaler.fit(master_train_df[input_cols])
print("✅ Input scaler (for height, weight) has been fitted.")

# Create one scaler for outputs
output_scaler = StandardScaler()
output_scaler.fit(master_train_df[output_cols])
print("✅ Output scaler (for 14 measurements) has been fitted.")

# --- 4. Save the Scalers to Files ---
# Scalers are saved in the current working directory, which is /kaggle/working/
joblib.dump(input_scaler, 'input_scaler.joblib')
joblib.dump(output_scaler, 'output_scaler.joblib')

print(f"\n✅ Scalers have been created and saved as 'input_scaler.joblib' and 'output_scaler.joblib'.")

In [ ]:
# ==========================================================
#          EXPERIMENT 3 - CELL 2: REDEFINE DATASET & DATALOADERS (FINAL FIX)
# ==========================================================
# ... (all the class definition code is unchanged) ...

# --- 3. Create all new DataLoaders for the experiment ---
print("\nCreating new DataLoaders with NormalizedBodyMDataset...")

# ... (data_root and dataset creation code is unchanged) ...

# Split into Train and Validation
train_size = int(0.8 * len(full_dataset_norm))
val_size = len(full_dataset_norm) - train_size
train_dataset_norm, val_dataset_norm = random_split(full_dataset_norm, [train_size, val_size])

print(f"  > Training samples:  {len(train_dataset_norm)}")
print(f"  > Validation samples: {len(val_dataset_norm)}")

# Create new DataLoaders
batch_size = 32
# FIX: Set num_workers to 0 to prevent deadlocks/stalls in CPU environment
train_dataloader_norm = DataLoader(train_dataset_norm, batch_size=batch_size, shuffle=True, num_workers=0) 
val_dataloader_norm = DataLoader(val_dataset_norm, batch_size=batch_size, shuffle=False, num_workers=0)

# Create new Test DataLoaders
# ... (test dataloader code is unchanged and already uses num_workers=0 by default) ...

print("\nAll normalized dataloaders are ready!")

In [ ]:
# ==========================================================
#          EXPERIMENT 3 - CELL 2: REDEFINE DATASET & DATALOADERS (KAGGLE FIX)
# ==========================================================
import torch
from torch.utils.data import Dataset, DataLoader, random_split
import pandas as pd
from PIL import Image
from torchvision import transforms
import os
import joblib
import numpy as np

# --- 1. Define Image Transformations (Unchanged) ---
data_transforms = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406],
                             std=[0.229, 0.224, 0.225])
])

# --- 2. Create the NEW "Normalized" Dataset Class (Unchanged Logic) ---
class NormalizedBodyMDataset(Dataset):
    def __init__(self, root_dir, transform=None):
        self.root_dir = root_dir
        self.transform = transform

        # --- NEW: Load the fitted scalers ---
        try:
            # Scalers were saved to the current directory
            self.input_scaler = joblib.load('input_scaler.joblib')
            self.output_scaler = joblib.load('output_scaler.joblib')
            print(f"NormalizedDataset (for {os.path.basename(root_dir)}): Scalers loaded successfully.")
        except FileNotFoundError:
            print("ERROR: Scaler files not found! Please run the 'CREATE SCALERS' cell first.")
            raise

        # --- Load CSVs (now correctly uses the passed in root_dir, which will be the Kaggle path) ---
        measurements_path = os.path.join(root_dir, 'measurements.csv')
        hwg_path = os.path.join(root_dir, 'hwg_metadata.csv')
        photo_map_path = os.path.join(root_dir, 'subject_to_photo_map.csv')

        measurements_df = pd.read_csv(measurements_path)
        hwg_df = pd.read_csv(hwg_path)
        photo_map_df = pd.read_csv(photo_map_path)

        # --- Filter (same as before) ---
        photo_counts = photo_map_df['subject_id'].value_counts()
        subjects_with_two_photos = photo_counts[photo_counts >= 2].index.tolist()
        master_df_unfiltered = pd.merge(measurements_df, hwg_df, on='subject_id')
        self.master_df = master_df_unfiltered[master_df_unfiltered['subject_id'].isin(subjects_with_two_photos)]
        self.photo_map_df = photo_map_df[photo_map_df['subject_id'].isin(subjects_with_two_photos)]

        # --- Define column lists ---
        self.input_cols = ['height_cm', 'weight_kg']
        self.output_cols = [
            'ankle', 'arm-length', 'bicep', 'calf', 'chest', 'forearm', 'height',
            'hip', 'leg-length', 'shoulder-breadth', 'shoulder-to-crotch',
            'thigh', 'waist', 'wrist'
        ]

    def __len__(self):
        return len(self.master_df)

    def __getitem__(self, idx):
        data_row = self.master_df.iloc[idx]
        subject_id = data_row['subject_id']

        # Get images (same as before)
        photo_ids = self.photo_map_df[self.photo_map_df['subject_id'] == subject_id]['photo_id'].tolist()
        photo_id_front, photo_id_side = photo_ids[0], photo_ids[1]
        
        # Paths now correctly use the self.root_dir, which points to the Kaggle folder
        img_path_front = os.path.join(self.root_dir, 'mask', f"{photo_id_front}.png")
        img_path_side = os.path.join(self.root_dir, 'mask_left', f"{photo_id_side}.png")
        
        image_front = Image.open(img_path_front).convert("RGB")
        image_side = Image.open(img_path_side).convert("RGB")

        if self.transform:
            image_front = self.transform(image_front)
            image_side = self.transform(image_side)

        # --- MODIFIED: Load, Normalize, and Convert ---
        height_val = data_row['height_cm']
        weight_val = data_row['weight_kg']
        measurements_vals = data_row[self.output_cols].values.astype('float32')

        inputs_np = np.array([[height_val, weight_val]], dtype='float32')
        measurements_np = measurements_vals.reshape(1, -1)

        norm_inputs = self.input_scaler.transform(inputs_np)
        norm_measurements = self.output_scaler.transform(measurements_np)

        norm_inputs_tensor = torch.tensor(norm_inputs[0], dtype=torch.float32)
        norm_measurements_tensor = torch.tensor(norm_measurements[0], dtype=torch.float32)
        # ------------------------------------------------

        sample = {
            'subject_id': subject_id,
            'image_front': image_front,
            'image_side': image_side,
            'height': norm_inputs_tensor[0], # Normalized height
            'weight': norm_inputs_tensor[1], # Normalized weight
            'measurements': norm_measurements_tensor # Normalized 14 measurements
        }
        return sample

# --- 3. Create all new DataLoaders for the experiment ---
print("\nCreating new DataLoaders with NormalizedBodyMDataset...")

# FIX: Use DATASET_ROOT for all data paths
# Create new FULL dataset (Training data)
data_root = os.path.join(DATASET_ROOT, 'train')
full_dataset_norm = NormalizedBodyMDataset(root_dir=data_root, transform=data_transforms)
print(f"✅ Full normalized dataset created with {len(full_dataset_norm)} samples.")

# Split into Train and Validation
train_size = int(0.8 * len(full_dataset_norm))
val_size = len(full_dataset_norm) - train_size
train_dataset_norm, val_dataset_norm = random_split(full_dataset_norm, [train_size, val_size])

print(f"  > Training samples:  {len(train_dataset_norm)}")
print(f"  > Validation samples: {len(val_dataset_norm)}")

# Create new DataLoaders
batch_size = 32
train_dataloader_norm = DataLoader(train_dataset_norm, batch_size=batch_size, shuffle=True, num_workers=2)
val_dataloader_norm = DataLoader(val_dataset_norm, batch_size=batch_size, shuffle=False, num_workers=2)

# Create new Test DataLoaders
testA_root = os.path.join(DATASET_ROOT, 'testA')
testB_root = os.path.join(DATASET_ROOT, 'testB')

testA_dataset_norm = NormalizedBodyMDataset(root_dir=testA_root, transform=data_transforms)
testA_dataloader_norm = DataLoader(testA_dataset_norm, batch_size=batch_size, shuffle=False)
print(f"✅ TestA normalized dataloader created with {len(testA_dataset_norm)} samples.")

testB_dataset_norm = NormalizedBodyMDataset(root_dir=testB_root, transform=data_transforms)
testB_dataloader_norm = DataLoader(testB_dataset_norm, batch_size=batch_size, shuffle=False)
print(f"✅ TestB normalized dataloader created with {len(testB_dataset_norm)} samples.")

print("\nAll normalized dataloaders are ready!")

In [ ]:
# ==========================================================
#          EXPERIMENT 3 - CELL 3: TRAIN ON NORMALIZED DATA (FINAL FIX)
# ==========================================================
import torch.optim as optim
import torch.nn as nn
import numpy as np
import time

# --- ADDED: WARNING SUPPRESSION ---
import warnings
# FIX: Removed 'from sklearn.exceptions import UserWarning'
# Use Python's built-in UserWarning for suppression
warnings.filterwarnings("ignore", category=UserWarning) 
print("✅ Suppressed repetitive Sklearn warnings.")
# -----------------------------------

print("--- STARTING NEW EXPERIMENT (TRAINING ON NORMALIZED DATA) ---")

# --- 1. Set up the Training Tools ---
# (We assume 'multi_input_model' is in memory, we are re-training it)
device = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")
multi_input_model.to(device)

# We will use MAE (L1Loss) again as it was better
criterion = nn.L1Loss()

# Reset the optimizer
optimizer = optim.Adam(list(multi_input_model.numerical_processor.parameters()) + list(multi_input_model.classifier.parameters()), lr=0.001)

# --- Early Stopping Setup (INCLUDED) ---
num_epochs = 20
patience = 3
patience_counter = 0
best_val_loss = np.inf
best_model_path = 'best_model_NORMALIZED_MAE.pth' # New save file

print(f"Starting training on {device}...")
print(f"Using MAE (L1Loss) on NORMALIZED data.")
print(f"Saving best model to {best_model_path}")
print(f"Will stop if validation loss does not improve for {patience} epochs.")

# --- 2. The Training Loop ---
for epoch in range(num_epochs):

    start_time = time.time() # <-- EPOCH TIMER START
    multi_input_model.train()
    running_loss = 0.0

    # === TRAINING (on _norm dataloader) ===
    for i, batch in enumerate(train_dataloader_norm):
        front_images = batch['image_front'].to(device)
        side_images = batch['image_side'].to(device)
        heights = batch['height'].to(device)
        weights = batch['weight'].to(device)
        measurements = batch['measurements'].to(device) # These are normalized

        optimizer.zero_grad()
        outputs = multi_input_model(front_images, side_images, heights, weights) # These are normalized
        loss = criterion(outputs, measurements)
        loss.backward()
        optimizer.step()
        running_loss += loss.item()

    epoch_train_loss = running_loss / len(train_dataloader_norm)

    # === VALIDATION (on _norm dataloader) ===
    multi_input_model.eval()
    val_loss = 0.0
    with torch.no_grad():
        for batch in val_dataloader_norm:
            front_images = batch['image_front'].to(device)
            side_images = batch['image_side'].to(device)
            heights = batch['height'].to(device)
            weights = batch['weight'].to(device)
            measurements = batch['measurements'].to(device) # Normalized

            outputs = multi_input_model(front_images, side_images, heights, weights) # Normalized
            loss = criterion(outputs, measurements) # This is normalized MAE
            val_loss += loss.item()

    epoch_val_loss = val_loss / len(val_dataloader_norm)
    epoch_time = time.time() - start_time # <-- EPOCH TIMER END

    # NOTE: This loss is the MAE of the *normalized* data, so it will be small (e.g., 0.6)
    print(f'Epoch {epoch + 1}/{num_epochs} | Time: {epoch_time:.2f}s | Train MAE (Norm): {epoch_train_loss:.3f} | Validation MAE (Norm): {epoch_val_loss:.3f}')

    # === Early Stopping Logic ===
    if epoch_val_loss < best_val_loss:
        print(f'Validation MAE improved ({best_val_loss:.3f} --> {epoch_val_loss:.3f}). Saving model... 💾')
        best_val_loss = epoch_val_loss
        patience_counter = 0
        torch.save(multi_input_model.state_dict(), best_model_path)
    else:
        patience_counter += 1
        print(f'Validation MAE did not improve. Patience {patience_counter}/{patience}')

    if patience_counter >= patience:
        print(f'\n--- 🛑 Early stopping triggered after {epoch + 1} epochs! ---')
        break

# --- 3. Load the Best Normalized Model ---
print(f"\nTraining complete. Loading best normalized model from {best_model_path}...")
multi_input_model.load_state_dict(torch.load(best_model_path))
print("✅ Best Normalized model loaded! Ready for final evaluation.")

In [ ]:
# ==========================================================
#       EXPERIMENT 3 - CELL 4: EVALUATE IN MILLIMETERS (mm)
# ==========================================================
import torch
import torch.nn as nn
import math
import joblib
import numpy as np

# --- 1. Load the "output" scaler to convert back to original units (cm) ---
try:
    output_scaler = joblib.load('output_scaler.joblib')
    print("Loaded 'output_scaler.joblib' for evaluation.")
except FileNotFoundError:
    print("ERROR: 'output_scaler.joblib' not found. Cannot convert results to cm/mm.")

# --- 2. Define a new evaluation function that converts to CM ---
def evaluate_normalized_model_in_cm(model, dataloader, device, output_scaler):
    model.eval()
    total_mae_loss_cm = 0.0
    total_rmse_loss_cm = 0.0
    num_samples = 0

    with torch.no_grad():
        for batch in dataloader:
            front_images = batch['image_front'].to(device)
            side_images = batch['image_side'].to(device)
            
            # 🛑 FIX: Extract heights and weights separately as the old model expects 4 arguments
            heights = batch['height'].to(device)
            weights = batch['weight'].to(device)
            
            # Get normalized measurements (ground truth)
            measurements_norm = batch['measurements'].to(device)

            # Get normalized predictions from the model
            # 🛑 FIX: Pass 4 separate arguments to the old multi_input_model
            outputs_norm = model(front_images, side_images, heights, weights)

            # --- CONVERT BACK TO CM ---
            # Move to CPU and convert to NumPy for the scaler
            outputs_norm_np = outputs_norm.cpu().numpy()
            measurements_norm_np = measurements_norm.cpu().numpy()

            # Use .inverse_transform() to get real CM values
            outputs_cm = output_scaler.inverse_transform(outputs_norm_np)
            measurements_cm = output_scaler.inverse_transform(measurements_norm_np)
            # ---------------------------

            # Calculate error on the CM values
            mae_loss_cm = np.mean(np.abs(outputs_cm - measurements_cm))
            mse_loss_cm = np.mean((outputs_cm - measurements_cm)**2)

            # Accumulate loss (multiply by batch size)
            batch_size = front_images.size(0)
            total_mae_loss_cm += mae_loss_cm * batch_size
            total_rmse_loss_cm += mse_loss_cm * batch_size # Note: this is MSE
            num_samples += batch_size

    avg_mae_cm = total_mae_loss_cm / num_samples
    avg_rmse_cm = math.sqrt(total_rmse_loss_cm / num_samples) # Sqrt of Avg MSE

    return avg_mae_cm, avg_rmse_cm

# --- 3. Run the Final Evaluation on the NORMALIZED Model ---
print("\n--- Starting Final Evaluation of the NORMALIZED Model ---")

# We use the model loaded in the cell above
# We use the new '_norm' dataloaders
device = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")

if 'testA_dataloader_norm' in locals():
    # 1. Get results in CM
    mae_testA_cm, rmse_testA_cm = evaluate_normalized_model_in_cm(multi_input_model, testA_dataloader_norm, device, output_scaler)

    # 2. Convert to MM
    mae_testA_mm = mae_testA_cm * 10
    rmse_testA_mm = rmse_testA_cm * 10

    print(f"\nResults for TestA (in-distribution):")
    print(f"  > MAE:  {mae_testA_mm:.3f} mm  <-- Compare this to 34.180 mm")
    print(f"  > RMSE: {rmse_testA_mm:.3f} mm")

if 'testB_dataloader_norm' in locals():
    # 1. Get results in CM
    mae_testB_cm, rmse_testB_cm = evaluate_normalized_model_in_cm(multi_input_model, testB_dataloader_norm, device, output_scaler)

    # 2. Convert to MM
    mae_testB_mm = mae_testB_cm * 10
    rmse_testB_mm = rmse_testB_cm * 10

    print(f"\nResults for TestB (in-the-wild):")
    print(f"  > MAE:  {mae_testB_mm:.3f} mm  <-- Compare this to 40.820 mm")
    print(f"  > RMSE: {rmse_testB_mm:.3f} mm")

In [ ]:
# ==========================================================
#       EXPERIMENT 4 - CELL 1: DEFINE AUGMENTATION
# ==========================================================
from torchvision import transforms

# --- 1. NEW Transforms for TRAINING (with Augmentation) ---
# We add random flips and rotations to fight overfitting
data_transforms_train = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.RandomHorizontalFlip(p=0.5), # 50% chance to flip the image
    transforms.RandomRotation(10),        # Randomly rotate between -10 and +10 degrees
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406],
                             std=[0.229, 0.224, 0.225])
])

# --- 2. OLD Transforms for VALIDATION/TESTING (no Augmentation) ---
# We use the original, non-random transforms for a fair test
data_transforms_val = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406],
                             std=[0.229, 0.224, 0.225])
])

print("✅ Experiment 4 Transforms defined:")
print("    > 'data_transforms_train' (with augmentation)")
print("    > 'data_transforms_val' (no augmentation)")

In [ ]:
# ==========================================================
#   EXPERIMENT 4 - CELL 2: CREATE AUGMENTED DATALOADERS (KAGGLE FIX)
# ==========================================================
from torch.utils.data import DataLoader, random_split
import os

print("--- Experiment 4: Creating Augmented DataLoaders ---")

# (We assume the 'NormalizedBodyMDataset' class is still in memory from Experiment 3)

# 1. Create the full dataset, using the VALIDATION transforms first
# FIX 1: Use DATASET_ROOT
data_root = os.path.join(DATASET_ROOT, 'train')
full_dataset_aug = NormalizedBodyMDataset(root_dir=data_root, transform=data_transforms_val)
print(f"✅ Full normalized dataset created with {len(full_dataset_aug)} samples.")

# 2. Split into Train and Validation
train_size = int(0.8 * len(full_dataset_aug))
val_size = len(full_dataset_aug) - train_size
train_dataset_aug, val_dataset_aug = random_split(full_dataset_aug, [train_size, val_size])

# --- 3. THIS IS THE KEY STEP ---
# We "reach inside" the training dataset and replace its transform
# with our new AUGMENTED transform.
train_dataset_aug.dataset.transform = data_transforms_train
print("✅ Applied 'data_transforms_train' (with augmentation) to the training set.")
# The 'val_dataset_aug' automatically keeps the original 'data_transforms_val'.

print(f"  > Training samples:  {len(train_dataset_aug)}")
print(f"  > Validation samples: {len(val_dataset_aug)}")

# 4. Create new DataLoaders
batch_size = 32
# FIX 2: Set num_workers=0 for stability
train_dataloader_aug = DataLoader(train_dataset_aug, batch_size=batch_size, shuffle=True, num_workers=0)
val_dataloader_aug = DataLoader(val_dataset_aug, batch_size=batch_size, shuffle=False, num_workers=0)

# 5. Create new Test DataLoaders (using the VALIDATION transform)
# FIX 1: Use DATASET_ROOT
testA_root = os.path.join(DATASET_ROOT, 'testA')
testB_root = os.path.join(DATASET_ROOT, 'testB')

testA_dataset_aug = NormalizedBodyMDataset(root_dir=testA_root, transform=data_transforms_val)
testA_dataloader_aug = DataLoader(testA_dataset_aug, batch_size=batch_size, shuffle=False)
print(f"✅ TestA augmented dataloader created with {len(testA_dataset_aug)} samples.")

testB_dataset_aug = NormalizedBodyMDataset(root_dir=testB_root, transform=data_transforms_val)
testB_dataloader_aug = DataLoader(testB_dataset_aug, batch_size=batch_size, shuffle=False)
print(f"✅ TestB augmented dataloader created with {len(testB_dataset_aug)} samples.")

print("\nAll augmented dataloaders are ready!")

In [ ]:
# ==========================================================
#   EXPERIMENT 4 - CELL 2: CREATE AUGMENTED DATALOADERS (FINAL FIX)
# ==========================================================
from torch.utils.data import DataLoader, random_split
import os # <-- Need os to join paths

print("--- Experiment 4: Creating Augmented DataLoaders ---")

# (We assume the 'NormalizedBodyMDataset' class is still in memory from Experiment 3)

# 1. Create the full dataset, using the VALIDATION transforms first
# 🛑 FIX: Use os.path.join(DATASET_ROOT, 'train') instead of '/content/dataset/train'
# Assuming DATASET_ROOT is defined from your initial setup cell
try:
    data_root = os.path.join(DATASET_ROOT, 'train')
except NameError:
    # Fallback if DATASET_ROOT was not set correctly, though this path is wrong for Kaggle
    data_root = '/kaggle/input/amazon-dataset/amazon-dataset/train' # <-- Try this if DATASET_ROOT is missing
    print("Warning: DATASET_ROOT not found. Using fallback path. Run your setup cell first.")

full_dataset_aug = NormalizedBodyMDataset(root_dir=data_root, transform=data_transforms_val)
print(f"✅ Full normalized dataset created with {len(full_dataset_aug)} samples.")

# 2. Split into Train and Validation
train_size = int(0.8 * len(full_dataset_aug))
val_size = len(full_dataset_aug) - train_size
train_dataset_aug, val_dataset_aug = random_split(full_dataset_aug, [train_size, val_size])

# --- 3. THIS IS THE KEY STEP ---
# We "reach inside" the training dataset and replace its transform
# with our new AUGMENTED transform.
train_dataset_aug.dataset.transform = data_transforms_train
print("✅ Applied 'data_transforms_train' (with augmentation) to the training set.")
# The 'val_dataset_aug' automatically keeps the original 'data_transforms_val'.

print(f"  > Training samples:  {len(train_dataset_aug)}")
print(f"  > Validation samples: {len(val_dataset_aug)}")

# 4. Create new DataLoaders
batch_size = 32
# 🛑 FIX: Set num_workers=0 for stability in the notebook environment
train_dataloader_aug = DataLoader(train_dataset_aug, batch_size=batch_size, shuffle=True, num_workers=0)
val_dataloader_aug = DataLoader(val_dataset_aug, batch_size=batch_size, shuffle=False, num_workers=0)

# 5. Create new Test DataLoaders (using the VALIDATION transform)
# 🛑 FIX: Use os.path.join(DATASET_ROOT, ...) for test paths
try:
    testA_root = os.path.join(DATASET_ROOT, 'testA')
    testB_root = os.path.join(DATASET_ROOT, 'testB')
except NameError:
    # Fallback for test paths
    testA_root = '/kaggle/input/amazon-dataset/amazon-dataset/testA'
    testB_root = '/kaggle/input/amazon-dataset/amazon-dataset/testB'


testA_dataset_aug = NormalizedBodyMDataset(root_dir=testA_root, transform=data_transforms_val)
# Set num_workers=0 for test dataloader stability
testA_dataloader_aug = DataLoader(testA_dataset_aug, batch_size=batch_size, shuffle=False, num_workers=0)
print(f"✅ TestA augmented dataloader created with {len(testA_dataset_aug)} samples.")

testB_dataset_aug = NormalizedBodyMDataset(root_dir=testB_root, transform=data_transforms_val)
# Set num_workers=0 for test dataloader stability
testB_dataloader_aug = DataLoader(testB_dataset_aug, batch_size=batch_size, shuffle=False, num_workers=0)
print(f"✅ TestB augmented dataloader created with {len(testB_dataset_aug)} samples.")

print("\nAll augmented dataloaders are ready!")

In [ ]:
# ==========================================================
# CELL 21: EXPERIMENT 4 - TRAINING WITH AUGMENTATION (KAGGLE FIX)
# ==========================================================
import time
import torch.optim as optim
import torch.nn as nn
from torch.optim.lr_scheduler import StepLR
import torch
import numpy as np
import torchvision.models as models # <-- Needed here
from torch.utils.data import DataLoader, random_split
import warnings
warnings.filterwarnings("ignore", category=UserWarning)

# --- 🛑 CRITICAL FIX: MultiInputMnasNet CLASS DEFINITION ---

NUM_BODY_MEASUREMENTS = 14

class MultiInputMnasNet(nn.Module):
    """
    Corrected custom model class. Included here to resolve previous TypeError.
    """
    def __init__(self, base_model, num_measurements=NUM_BODY_MEASUREMENTS):
        super(MultiInputMnasNet, self).__init__()

        self.base_model = base_model

        self.feature_extractor = nn.Sequential(
            *list(base_model.children())[:-1]
        )

        self.avgpool = nn.AdaptiveAvgPool2d((1, 1))
        feature_dim = 1280

        numerical_feature_dim = 64
        self.numerical_processor = nn.Sequential(
            nn.Linear(2, 32),
            nn.ReLU(),
            nn.Linear(32, numerical_feature_dim),
            nn.ReLU()
        )

        combined_dim = feature_dim * 2 + numerical_feature_dim

        self.classifier = nn.Sequential(
            nn.Linear(combined_dim, 1024),
            nn.ReLU(),
            nn.Dropout(0.3),
            nn.Linear(1024, 512),
            nn.ReLU(),
            nn.Linear(512, num_measurements)
        )


    def forward(self, frontal_img, lateral_img, height_weight):
        f_features = self.feature_extractor(frontal_img)
        f_features = self.avgpool(f_features)
        f_features = torch.flatten(f_features, 1)

        l_features = self.feature_extractor(lateral_img)
        l_features = self.avgpool(l_features)
        l_features = torch.flatten(l_features, 1)

        n_features = self.numerical_processor(height_weight)

        combined_features = torch.cat((f_features, l_features, n_features), dim=1)

        outputs = self.classifier(combined_features)

        return outputs

# ------------------------------------------------------------------
# Training Script Starts Here
# ------------------------------------------------------------------

# Define necessary variables
device = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")

# Define the MnasNet backbone
print("Initializing MnasNet backbone...")
# 🛑 FIX: Initialize without pre-trained weights to avoid network connection
# If you want pre-trained weights, you must enable internet in your Kaggle notebook settings.
mnasnet_base = models.mnasnet1_0(weights=None) 

for param in mnasnet_base.parameters():
    param.requires_grad = False
print("MnasNet backbone loaded and frozen (Training will start from scratch for the new layers).")

# --- Hyperparameters ---
num_epochs = 50
patience = 7
best_model_path_aug = 'mnasnet_best_aug.pt' 

# Re-initialize the model
multi_input_model_aug = MultiInputMnasNet(mnasnet_base).to(device)

# --- Optimizer and Loss ---
optimizer = optim.Adam(
    list(multi_input_model_aug.numerical_processor.parameters()) +
    list(multi_input_model_aug.classifier.parameters()),
    lr=0.001
)
criterion = nn.L1Loss()
scheduler = StepLR(optimizer, step_size=10, gamma=0.1)

# --- Training Loop ---
best_val_loss = float('inf')
epochs_no_improve = 0
start_time = time.time()

print(f"Starting Training for Experiment 4 (MAE + Norm + Augmentation) on device: {device}\n")

for epoch in range(num_epochs):
    multi_input_model_aug.train()
    train_loss = 0.0

    # Training loop
    for batch in train_dataloader_aug:
        # Transfer data to device
        frontal_img = batch['image_front'].to(device)
        lateral_img = batch['image_side'].to(device)
        # Combine height and weight into a single tensor for the numerical processor
        height_weight = torch.cat([batch['height'].unsqueeze(1), batch['weight'].unsqueeze(1)], dim=1).to(device)
        measurements = batch['measurements'].to(device)

        # Zero the parameter gradients
        optimizer.zero_grad()

        # Forward pass
        outputs = multi_input_model_aug(frontal_img, lateral_img, height_weight)

        # Compute loss (L1 Loss is MAE)
        loss = criterion(outputs, measurements)
        loss.backward()
        optimizer.step()
        
        # Multiply loss by batch size before summing to get total loss
        train_loss += loss.item() * frontal_img.size(0) 

    # --- Validation Phase ---
    multi_input_model_aug.eval()
    val_loss = 0.0
    with torch.no_grad():
        # Validation loop
        for batch in val_dataloader_aug:
            frontal_img = batch['image_front'].to(device)
            lateral_img = batch['image_side'].to(device)
            # Combine height and weight into a single tensor
            height_weight = torch.cat([batch['height'].unsqueeze(1), batch['weight'].unsqueeze(1)], dim=1).to(device)
            measurements = batch['measurements'].to(device)

            outputs = multi_input_model_aug(frontal_img, lateral_img, height_weight)
            loss = criterion(outputs, measurements)
            # Multiply loss by batch size before summing to get total loss
            val_loss += loss.item() * frontal_img.size(0)

    # --- Calculate Epoch Metrics ---
    # Divide total accumulated loss by total number of samples
    epoch_train_loss = train_loss / len(train_dataloader_aug.dataset)
    epoch_val_loss = val_loss / len(val_dataloader_aug.dataset)
    scheduler.step()
    elapsed = time.time() - start_time

    print(f"Epoch [{epoch+1:03d}/{num_epochs}] | Time: {elapsed:.2f}s | "
          f"Train MAE: {epoch_train_loss:.4f} | Val MAE: {epoch_val_loss:.4f}")

    # --- Early Stopping and Model Saving ---
    if epoch_val_loss < best_val_loss:
        best_val_loss = epoch_val_loss
        torch.save(multi_input_model_aug.state_dict(), best_model_path_aug)
        epochs_no_improve = 0
        print(f"💾 Best model updated (Val MAE improved to {best_val_loss:.4f})")
    else:
        epochs_no_improve += 1
        if epochs_no_improve == patience:
            print(f"\n🛑 Early stopping triggered after {patience} epochs with no improvement.")
            break

print(f"\n✅ Training complete. Loading best BMNet model from: {best_model_path_aug}")
# ⚠️ This line assumes the model was saved successfully at least once.
multi_input_model_aug.load_state_dict(torch.load(best_model_path_aug))
print("Model loaded and ready for final evaluation.")

In [ ]:
# ==========================================================
#       EXPERIMENT 4 - CELL 4: EVALUATE IN MILLIMETERS (mm)
# ==========================================================
import torch
import torch.nn as nn
import math
import joblib
import numpy as np

# --- 1. Load the "output" scaler to convert back to original units (cm) ---
try:
    output_scaler = joblib.load('output_scaler.joblib')
    print("Loaded 'output_scaler.joblib' for evaluation.")
except FileNotFoundError:
    print("ERROR: 'output_scaler.joblib' not found. Cannot convert results to cm/mm.")

# (We assume the 'evaluate_normalized_model_in_cm' function is still in memory)
# Let's redefine it just in case your session reset
def evaluate_normalized_model_in_cm(model, dataloader, device, output_scaler):
    model.eval()
    total_mae_loss_cm = 0.0
    total_rmse_loss_cm = 0.0
    num_samples = 0

    with torch.no_grad():
        for batch in dataloader:
            front_images = batch['image_front'].to(device)
            side_images = batch['image_side'].to(device)
            # Combine height and weight into a single tensor
            height_weight = torch.cat([batch['height'].unsqueeze(1), batch['weight'].unsqueeze(1)], dim=1).to(device)
            
            measurements_norm = batch['measurements'].to(device)
            outputs_norm = model(front_images, side_images, height_weight)

            # --- CONVERT BACK TO CM ---
            outputs_norm_np = outputs_norm.cpu().numpy()
            measurements_norm_np = measurements_norm.cpu().numpy()
            outputs_cm = output_scaler.inverse_transform(outputs_norm_np)
            measurements_cm = output_scaler.inverse_transform(measurements_norm_np)
            # ---------------------------

            mae_loss_cm = np.mean(np.abs(outputs_cm - measurements_cm))
            mse_loss_cm = np.mean((outputs_cm - measurements_cm)**2)

            batch_size = front_images.size(0)
            total_mae_loss_cm += mae_loss_cm * batch_size
            total_rmse_loss_cm += mse_loss_cm * batch_size # MSE
            num_samples += batch_size

    avg_mae_cm = total_mae_loss_cm / num_samples
    avg_rmse_cm = math.sqrt(total_rmse_loss_cm / num_samples) # Sqrt of Avg MSE

    return avg_mae_cm, avg_rmse_cm

# --- 3. Run the Final Evaluation on the AUGMENTED Model ---
print("\n--- Starting Final Evaluation of the AUGMENTED Model ---")
device = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")

# Ensure multi_input_model_aug is defined by running the training cell (Cell 21) first!
# We use the new '_aug' dataloaders

if 'testA_dataloader_aug' in locals():
    # 1. Get results in CM
    mae_testA_cm, rmse_testA_cm = evaluate_normalized_model_in_cm(multi_input_model_aug, testA_dataloader_aug, device, output_scaler)

    # 2. Convert to MM
    mae_testA_mm = mae_testA_cm * 10
    rmse_testA_mm = rmse_testA_cm * 10

    print(f"\nResults for TestA (in-distribution):")
    print(f"  > MAE:  {mae_testA_mm:.3f} mm  ")
    print(f"  > RMSE: {rmse_testA_mm:.3f} mm")

if 'testB_dataloader_aug' in locals():
    # 1. Get results in CM
    mae_testB_cm, rmse_testB_cm = evaluate_normalized_model_in_cm(multi_input_model_aug, testB_dataloader_aug, device, output_scaler)

    # 2. Convert to MM
    mae_testB_mm = mae_testB_cm * 10
    rmse_testB_mm = rmse_testB_cm * 10

    print(f"\nResults for TestB (in-the-wild):")
    print(f"  > MAE:  {mae_testB_mm:.3f} mm ")
    print(f"  > RMSE: {rmse_testB_mm:.3f} mm")

In [ ]:
# ==========================================================
# CELL 22: EVALUATE EXPERIMENT 4 ON TEST SETS (KAGGLE FIX)
# ==========================================================
import torch
import numpy as np
import joblib
import os
import math # Needed for square root in evaluation function if it wasn't re-defined later

# Define necessary variables (Must match path defined in Cell 6)
device = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")
# FIX: Use local file name
best_model_path_aug = 'mnasnet_best_aug.pt'

# 1. Load the model and scaler
try:
    # Ensure the model is defined and on device (defined in Cell 21)
    multi_input_model_aug.load_state_dict(torch.load(best_model_path_aug, map_location=device))
    multi_input_model_aug.to(device)
    multi_input_model_aug.eval()

    # Load the output scaler (Must have run the data loading cell)
    output_scaler = joblib.load('output_scaler.joblib')
    print("Model and Scaler loaded for evaluation.")

except NameError as e:
    print(f"Error: {e}. Did you run the data loader cell and Cell 21 completely?")
    raise
except FileNotFoundError as e:
    print(f"Error: {e}. Check your model save path and file names.")
    raise

def evaluate_normalized_model_in_cm(model, dataloader, device, output_scaler):
    """Evaluates model and returns MAE (cm) and RMSE (cm) using un-normalized values."""
    all_preds_norm = []
    all_targets_norm = []

    with torch.no_grad():
        for batch in dataloader:
            frontal_img = batch['image_front'].to(device)
            lateral_img = batch['image_side'].to(device)
            # Combine height and weight into a single tensor
            height_weight = torch.cat([batch['height'].unsqueeze(1), batch['weight'].unsqueeze(1)], dim=1).to(device)
            measurements = batch['measurements'].to(device) # Normalized targets

            outputs = model(frontal_img, lateral_img, height_weight)

            all_preds_norm.append(outputs.cpu().numpy())
            all_targets_norm.append(measurements.cpu().numpy())

    preds_norm = np.concatenate(all_preds_norm, axis=0)
    targets_norm = np.concatenate(all_targets_norm, axis=0)

    # 1. Inverse transform to get predictions in original units (cm)
    preds_cm = output_scaler.inverse_transform(preds_norm)
    targets_cm = output_scaler.inverse_transform(targets_norm)

    # 2. Calculate MAE and RMSE in CM
    mae_cm = np.mean(np.abs(preds_cm - targets_cm))
    rmse_cm = np.sqrt(np.mean((preds_cm - targets_cm)**2))

    return mae_cm, rmse_cm

# --- Evaluation on Test A ---
print("\n--- Evaluating Test A (In-Lab) ---")
# 1. Get results in CM
mae_testA_cm_exp4, rmse_testA_cm_exp4 = evaluate_normalized_model_in_cm(multi_input_model_aug, testA_dataloader_aug, device, output_scaler)

# 2. Convert to MM and set variable for plotting
mae_testA_mm_exp4 = mae_testA_cm_exp4 * 10
rmse_testA_mm_exp4 = rmse_testA_cm_exp4 * 10

print(f"Results for Test A (Exp 4):")
print(f"  > MAE:  {mae_testA_mm_exp4:.3f} mm")
print(f"  > RMSE: {rmse_testA_mm_exp4:.3f} mm")


# --- Evaluation on Test B ---
print("\n--- Evaluating Test B (In-The-Wild) ---")
# 1. Get results in CM
mae_testB_cm_exp4, rmse_testB_cm_exp4 = evaluate_normalized_model_in_cm(multi_input_model_aug, testB_dataloader_aug, device, output_scaler)

# 2. Convert to MM and set variable for plotting
mae_testB_mm_exp4 = mae_testB_cm_exp4 * 10
rmse_testB_mm_exp4 = rmse_testB_cm_exp4 * 10

print(f"Results for Test B (Exp 4):")
print(f"  > MAE:  {mae_testB_mm_exp4:.3f} mm")
print(f"  > RMSE: {rmse_testB_mm_exp4:.3f} mm")

print("\n✅ Evaluation complete. Variables for Cell 23 are set.")

**Testing**